# 1. Chuẩn bị dữ liệu - VOYA_VSL Dataset

**Mục tiêu**: Download và xử lý VOYA_VSL dataset từ HuggingFace.

| Thông tin | Chi tiết |
|-----------|---------|
| **Dataset** | VOYA_VSL (HuggingFace) |
| **Số class** | 161 classes (ký hiệu tiếng Việt) |
| **Mỗi sample** | (60 frames, 1605 features) - MediaPipe keypoints |
| **Feature layout** | Pose(99) + LeftHand(63) + RightHand(63) + Face(1380) = 1605 |
| **Output** | Chỉ lấy hand landmarks: 126 features/frame |

**Hỗ trợ chạy trên**: Google Colab, Kaggle, hoặc Local

> **Lưu ý**: Trên Colab/Kaggle, data và models sẽ được lưu vào **Google Drive** để không mất khi session kết thúc. Thay đổi `NUM_CLASSES` bên dưới để download nhiều hoặc ít classes hơn.

## 0. Setup môi trường (Colab / Kaggle / Local)

Cell bên dưới sẽ tự động:
1. Detect môi trường (Colab, Kaggle, hay Local)
2. Mount Google Drive (trên Colab/Kaggle) để lưu data & models
3. Clone repo nếu cần
4. Cài dependencies

**Trên Google Drive**, project sẽ nằm tại: `My Drive/vsl-recognition/`

In [ ]:
# === AUTO-DETECT MÔI TRƯỜNG ===
import os, sys

ENV = "local"
DRIVE_ROOT = ""

# --- Google Colab ---
if "google.colab" in sys.modules or os.path.exists("/content"):
    ENV = "colab"
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/vsl-recognition"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Google Colab detected. Drive mounted at: {DRIVE_ROOT}")

# --- Kaggle ---
elif os.path.exists("/kaggle"):
    ENV = "kaggle"
    # Kaggle: cần bật "Google Drive" trong Add-ons hoặc dùng output
    # Nếu đã mount Drive qua Kaggle Add-on:
    if os.path.exists("/root/.kaggle") or os.path.exists("/kaggle/working"):
        # Thử mount Drive nếu có thể
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            DRIVE_ROOT = "/content/drive/MyDrive/vsl-recognition"
        except ImportError:
            DRIVE_ROOT = "/kaggle/working/vsl-recognition"
        os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Kaggle detected. Output at: {DRIVE_ROOT}")

# --- Local ---
else:
    # Giả sử đang chạy từ thư mục notebooks/
    DRIVE_ROOT = os.path.dirname(os.path.abspath("."))
    if os.path.basename(os.getcwd()) == "notebooks":
        DRIVE_ROOT = os.path.dirname(os.getcwd())
    print(f"Local environment. Project root: {DRIVE_ROOT}")

# Tạo thư mục cần thiết
DATA_ROOT = os.path.join(DRIVE_ROOT, "data", "processed")
MODELS_ROOT = os.path.join(DRIVE_ROOT, "models")
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(MODELS_ROOT, exist_ok=True)

print(f"\nMôi trường: {ENV}")
print(f"Data sẽ lưu tại: {DATA_ROOT}")
print(f"Models sẽ lưu tại: {MODELS_ROOT}")

In [ ]:
# === CẤU HÌNH ===
NUM_CLASSES = 10          # Số class muốn download (max 161, dùng 10 để test nhanh)
OUTPUT_DIR = DATA_ROOT    # Tự động từ cell setup (Drive hoặc local)
FEATURE_MODE = "hands_only"  # "hands_only" (126) | "hands_and_pose" (225) | "full" (1605)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
!pip install -q huggingface_hub numpy

import os, json
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files

# Feature layout trong 1605-dim vector (MediaPipe Holistic)
POSE_START, POSE_END = 0, 99         # 33 landmarks × 3
LEFT_HAND_START, LEFT_HAND_END = 99, 162   # 21 landmarks × 3
RIGHT_HAND_START, RIGHT_HAND_END = 162, 225  # 21 landmarks × 3
FACE_START, FACE_END = 225, 1605     # 460 landmarks × 3

def extract_hand_features(sequence):
    """Lấy chỉ hand landmarks: (T, 1605) → (T, 126)"""
    left = sequence[:, LEFT_HAND_START:LEFT_HAND_END]
    right = sequence[:, RIGHT_HAND_START:RIGHT_HAND_END]
    return np.concatenate([left, right], axis=-1)

def extract_hand_and_pose_features(sequence):
    """Lấy hand + pose: (T, 1605) → (T, 225)"""
    pose = sequence[:, POSE_START:POSE_END]
    left = sequence[:, LEFT_HAND_START:LEFT_HAND_END]
    right = sequence[:, RIGHT_HAND_START:RIGHT_HAND_END]
    return np.concatenate([pose, left, right], axis=-1)

print("Setup done!")

## 1.1 Download dataset từ HuggingFace

In [ ]:
# Liệt kê files trên HuggingFace
all_files = list_repo_files("Kateht/VOYA_VSL", repo_type="dataset")
npz_files = sorted([f for f in all_files if f.endswith(".npz")])
print(f"Tổng số class có sẵn: {len(npz_files)}")
print(f"Sẽ download: {min(NUM_CLASSES, len(npz_files))} classes")

# Download labels
labels_path = hf_hub_download("Kateht/VOYA_VSL", "labels.json", repo_type="dataset")
with open(labels_path, "r", encoding="utf-8") as f:
    all_labels = json.load(f)

# Xem một vài labels
for k, v in list(all_labels.items())[:10]:
    print(f"  {k}: {v}")

## 1.2 Download và xử lý từng class

In [ ]:
extract_fn = {
    "hands_only": extract_hand_features,
    "hands_and_pose": extract_hand_and_pose_features,
    "full": lambda x: x,
}[FEATURE_MODE]

static_dir = os.path.join(OUTPUT_DIR, "static")
dynamic_dir = os.path.join(OUTPUT_DIR, "dynamic")
os.makedirs(static_dir, exist_ok=True)
os.makedirs(dynamic_dir, exist_ok=True)

label_mapping = {}
total_samples = 0
num_to_download = min(NUM_CLASSES, len(npz_files))

for i, npz_file in enumerate(npz_files[:num_to_download]):
    class_key = npz_file.replace("Merged/", "").replace(".npz", "")
    class_name = all_labels.get(class_key, class_key)
    safe_name = class_name.replace("/", "_").replace("\\", "_").replace(" ", "_")
    safe_name = f"{i:04d}_{safe_name}"

    print(f"[{i+1}/{num_to_download}] {class_key}: {class_name} ", end="")

    try:
        path = hf_hub_download("Kateht/VOYA_VSL", npz_file, repo_type="dataset")
        data = np.load(path)
        sequences = data["sequences"]  # (N, 60, 1605)

        processed = np.array([extract_fn(s) for s in sequences])

        # Lưu dynamic sequences
        dyn_dir = os.path.join(dynamic_dir, safe_name)
        os.makedirs(dyn_dir, exist_ok=True)
        for j in range(len(processed)):
            np.save(os.path.join(dyn_dir, f"seq_{j:04d}.npy"), processed[j])

        # Lưu static (frame giữa)
        static_cls_dir = os.path.join(static_dir, safe_name)
        os.makedirs(static_cls_dir, exist_ok=True)
        for j in range(len(processed)):
            np.save(os.path.join(static_cls_dir, f"sample_{j:04d}.npy"), processed[j][30])

        label_mapping[str(i)] = class_name
        total_samples += len(processed)
        print(f"→ {len(processed)} samples ({processed[0].shape})")
    except Exception as e:
        print(f"ERROR: {e}")

print(f"\n✓ Done! {len(label_mapping)} classes, {total_samples} samples")

## 1.3 Lưu metadata

In [ ]:
meta = {
    "labels": label_mapping,
    "feature_mode": FEATURE_MODE,
    "num_classes": len(label_mapping),
    "total_samples": total_samples,
    "sequence_length": 60,
}
meta_path = os.path.join(OUTPUT_DIR, "meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"Metadata saved to {meta_path}")
print(f"\nLabel mapping:")
for k, v in label_mapping.items():
    print(f"  {k}: {v}")

## 1.4 Kiểm tra dữ liệu đã xử lý

In [ ]:
import matplotlib.pyplot as plt

# Đếm samples mỗi class
class_counts = {}
for cls_dir in sorted(os.listdir(static_dir)):
    cls_path = os.path.join(static_dir, cls_dir)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.endswith(".npy")])
        class_counts[cls_dir.split("_", 1)[1]] = count

# Biểu đồ phân phối
plt.figure(figsize=(12, 5))
plt.bar(range(len(class_counts)), class_counts.values())
plt.xticks(range(len(class_counts)), class_counts.keys(), rotation=45, ha="right")
plt.ylabel("Số samples")
plt.title("Phân phối samples theo class")
plt.tight_layout()
plt.show()

# Visualize 1 sample
sample_dir = os.path.join(static_dir, sorted(os.listdir(static_dir))[0])
sample = np.load(os.path.join(sample_dir, os.listdir(sample_dir)[0]))
print(f"\nSample shape: {sample.shape}")
print(f"Min: {sample.min():.4f}, Max: {sample.max():.4f}, Mean: {sample.mean():.4f}")

---
**Tiếp theo**: Chạy notebook `02_train_static_model.ipynb` để train CNN-1D cho static signs.